In [ ]:
import pandas as pd

In [2]:
# Text Processing Libraries
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
from collections import Counter
from nltk import ngrams

In [3]:
# Sentiment Analysis
from leia import SentimentIntensityAnalyzer

In [4]:
# # Download required NLTK data
# nltk.download('stopwords')
# nltk.download('punkt')

## Criando a base de dados

In [5]:
df = pd.read_csv("Data/olist_order_reviews_dataset.csv")

In [6]:
# Selecting only necessary columns for NLP analysis
nlp_df = df[['review_comment_title', 'review_comment_message']]

## Limpeza de dados

In [7]:
def remove_duplicates_nlp_df(nlp_df, column_name='review_comment_message'):
    
    # Remove duplicates based on the specified column, keeping the first occurrence
    nlp_df = nlp_df.drop_duplicates(subset=[column_name], keep='first').reset_index(drop=True)
    
    # Display the total entries after removing duplicates
    print(f"Total entries after removing duplicates in '{column_name}': {nlp_df.shape[0]}")
    
    return nlp_df

# Remove duplicates from 'nlp_df' based on the 'review_comment_message' column
nlp_df = remove_duplicates_nlp_df(nlp_df, 'review_comment_message')

# Display the first few records to verify
nlp_df.head()

Total entries after removing duplicates in 'review_comment_message': 36922


,review_comment_title,review_comment_message
0,NaN,NaN
1,NaN,Recebi bem antes do prazo estipulado.
2,NaN,Parabéns lojas lannister adorei comprar pela I...
3,recomendo,aparelho eficiente. no site a marca do aparelh...
4,NaN,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"


In [ ]:
def clean_reviews(df):
    
    # Remove rows where 'review_comment_message' is empty
    df = df.dropna(subset=['review_comment_message']).reset_index(drop=True)

    # Remove duplicate rows
    df = df.drop_duplicates(subset=['review_comment_message'])

    return df

# Assuming 'nlp_df' is your dataframe
df_cleaned = clean_reviews(nlp_df)

# Display the first records to check
df_cleaned.head()

,review_comment_title,review_comment_message
0,recomendo,aparelho eficiente. no site a marca do aparelh...
1,Super recomendo,"Vendedor confiável, produto ok e entrega antes..."
2,Não chegou meu produto,Péssimo
3,Ótimo,Loja nota 10
4,Muito bom.,Recebi exatamente o que esperava. As demais en...


## Processamento do texto

In [9]:
# # Define Portuguese stopwords
# STOP_WORDS = set(stopwords.words('portuguese'))

# # Helper function to clean and tokenize text
# def clean_and_tokenize(text):
#     # Ensure the text is a string
#     if not isinstance(text, str):
#         return "", []
    
#     # Convert to lowercase, remove punctuation, and split into words
#     cleaned_text = text.lower().translate(str.maketrans('', '', string.punctuation))
#     words = cleaned_text.split()
    
#     # Remove stopwords and create tokens
#     filtered_words = [word for word in words if word not in STOP_WORDS]
#     return " ".join(filtered_words), filtered_words

# # Main function to preprocess and clean the dataframe
# def preprocess_nlp_df(df):
#     # Clean, remove stopwords, and tokenize comments
#     df[['review_comment_message_clean', 'review_comment_message_tokens']] = df['review_comment_message'].apply(
#         lambda text: pd.Series(clean_and_tokenize(text))
#     )
    
#     # Remove rows with NaN values in key columns
#     df.dropna(subset=['review_comment_title', 'review_comment_message'], inplace=True)
    
#     # Drop duplicate rows based on the 'review_comment_message' and 'review_comment_title' columns
#     df.drop_duplicates(subset=['review_comment_message', 'review_comment_title'], inplace=True)
    
#     return df.reset_index(drop=True)

# # Preprocess the dataset 
# nlp_df = preprocess_nlp_df(nlp_df)

# # Display the first records to check
# nlp_df[['review_comment_message', 'review_comment_message_clean', 'review_comment_message_tokens']].head()

## Processamento de linguagem natural

In [ ]:
# Initialize the Sentiment Analyzer once
analyzer = SentimentIntensityAnalyzer()

def classify_sentiment(df, column_name='review_comment_message_clean'):
    # Vectorized function to get sentiment classification
    def get_sentiment_classification(text):
        scores = analyzer.polarity_scores(text)
        if scores['compound'] >= 0.05:
            return 'Positive'
        elif scores['compound'] <= -0.05:
            return 'Negative'
        else:
            return 'Neutral'
        
    def get_sentiment_score(text):
        scores = analyzer.polarity_scores(text)
        return scores['compound']
    
    # Apply sentiment analysis using map for faster iteration
    df[f'{column_name}_sentiment'] = df[column_name].map(get_sentiment_classification)
    df[f'{column_name}_score'] = df[column_name].map(get_sentiment_score)
    return df

# Classify sentiment in 'nlp_df' based on the 'review_comment_message_clean' column
nlp_df = nlp_df.dropna(subset=["review_comment_message"]).copy(); 
nlp_df = classify_sentiment(nlp_df, 'review_comment_message')

# Display the sentiment results
nlp_df[['review_comment_message', 'review_comment_message_sentiment']].head(25)

C:\Users\super\AppData\Local\Temp\ipykernel_1276\2226216941.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{column_name}_sentiment'] = df[column_name].map(get_sentiment_classification)
C:\Users\super\AppData\Local\Temp\ipykernel_1276\2226216941.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{column_name}_score'] = df[column_name].map(get_sentiment_score)


,review_comment_message,review_comment_message_sentiment
1,Recebi bem antes do prazo estipulado.,Positive
2,Parabéns lojas lannister adorei comprar pela I...,Positive
3,aparelho eficiente. no site a marca do aparelh...,Positive
4,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",Positive
5,"Vendedor confiável, produto ok e entrega antes...",Positive
6,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...",Neutral
7,Péssimo,Neutral
8,Loja nota 10,Neutral
9,obrigado pela atençao amim dispensada,Positive
10,A compra foi realizada facilmente.\r\nA entreg...,Positive


In [13]:
nlp_df[['review_comment_message', 'review_comment_message_sentiment', 'review_comment_message_score']].tail(25)

,review_comment_message,review_comment_message_sentiment,review_comment_message_score
36897,"Amei , indico a todos do ML\r\n",Neutral,0.0000
36898,"Entrega bem antes do prazo estipulado, produto...",Positive,0.7351
36899,"Produto lindo, entrega rápida e segura.",Neutral,0.0000
36900,Eu recebi o seguinte email e preciso saber com...,Positive,0.5242
36901,Cortina que dá um toque de requinte para qualq...,Positive,0.7239
36902,Super antes do prazo!!!,Positive,0.6981
36903,"Recomendo, compra segura entrega correta.",Positive,0.3612
36904,"Boa tarde, gostei de ter comprado, o material ...",Positive,0.8625
36905,Boa tarde. \r\nNão recebo todos os produtos fa...,Positive,0.6808
36906,ótimo serviço e ótimo produto muito satisfeita!,Positive,0.6476


In [15]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [16]:
nlp_df.to_excel("tabela.xlsx", index=False)

In [17]:
nlp_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36921 entries, 1 to 36921
Data columns (total 4 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   review_comment_title              8865 non-null   object 
 1   review_comment_message            36921 non-null  object 
 2   review_comment_message_sentiment  36921 non-null  object 
 3   review_comment_message_score      36921 non-null  float64
dtypes: float64(1), object(3)
memory usage: 1.4+ MB


In [ ]:
texto_bruto = "Esse produto é HORRIVEL!!! NAO Recomendo 👍😊"
score = analyzer.polarity_scores(texto_bruto)
print(score['compound'])  # Score já processado!